# Credit Metrics

### Create Credit Metrics Model from Input data

In [1]:
import sys
sys.path.append("../../..")  # RiVaPy root directory

In [2]:
import pandas as pd
import numpy as np
import json
from rivapy.instruments.components import Issuer
from rivapy.credit.creditmetrics import CreditMetricsModel
import plotly.express as px

# load
positions = pd.read_csv("Input/positions.csv")
issuer_df = pd.read_csv("Input/issuers.csv")
stock_data = pd.read_csv("Input/stock_data.csv", parse_dates=["Date"])
TransMat = np.matrix(np.loadtxt("Input/transition_matrix.csv", delimiter=","))

with open("Input/cm_config.json", "r", encoding="utf-8") as f:
    cfg = json.load(f)
list_of_indices = cfg["list_of_indices"]
mapping_countries_on_indices = cfg["mapping_countries_on_indices"]

# convert issuer_df rows back to Issuer objects (adjust constructor args if needed)
issuer_list = []
for _, row in issuer_df.iterrows():
    issuer_list.append(
        Issuer(
            obj_id=row["IssuerID"],
            name=row["IssuerName"],
            rating=row["Rating"],
            country=row["Country"],
            sector=row.get("Sector", ""),
            esg_rating=row.get("ESG_Rating", ""),
        )
    )

# instantiate model (use the same parameters you used originally)
cm = CreditMetricsModel(
    n_simulation=5000,
    transition_matrix=TransMat,
    position_data=positions,
    issuer_data=issuer_list,
    stock_data=stock_data,
    r=0.0,
    t=1.0,
    confidencelevel=1.0,
    seed=42,
    list_of_indices=list_of_indices,
    mapping_countries_on_indices=mapping_countries_on_indices,
)

c:\Users\Anwender\source\repos\RiVaPy\notebooks\credit\test_data\../../..\rivapy\__init__.py:13: UserWarning: The pyvacon module is not available. You may not use all functionality without this module. Consider installing pyvacon.
  warnings.warn("The pyvacon module is not available. You may not use all functionality without this module. Consider installing pyvacon.")


In [3]:
np.set_printoptions(formatter={'float': '{: 0.5f}'.format})

#### Calculate VaR

In [8]:
mc_scenarios, rr_scenarios, issuer_ids, issuer_names = cm.mc_calculation()
print("mc_scenarios shape:", getattr(mc_scenarios, "shape", None))
print("issuer_ids:", issuer_ids)

mc_scenarios shape: (5000, 5)
issuer_ids: ['AAPL' 'BASF' 'BNP' 'GM' 'VOW']


c:\Users\Anwender\source\repos\RiVaPy\notebooks\credit\test_data\../../..\rivapy\credit\creditmetrics.py:119: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = mergedData.pct_change()


In [9]:
mc_scenarios.describe()

,AAPL,BASF,BNP,GM,VOW
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,-271.804500,-4378.643250,-779.256000,-10459.148800,-371.628000
std,11776.658699,39671.700955,19045.281385,69484.902259,12904.161348
min,-824917.500000,-636352.500000,-599640.000000,-633216.000000,-899460.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,0.000000,0.000000
max,0.000000,1083.750000,300.000000,6720.000000,450.000000


In [10]:
loss_distribution = cm.get_loss_distribution(mc_scenarios)

In [11]:
portfolio_VaR = cm.get_portfolio_VaR(loss_distribution)
portfolio_VaR

np.float64(633628.5)

#### Calculate Expected Shortfall

In [12]:
cm.get_portfolio_ES(loss_distribution)

np.float64(718633.3)

#### Display Loss Distribution

In [13]:
px.histogram(loss_distribution)